In [13]:
import git
import sys

repo = git.Repo(search_parent_directories=True)
sys.path.append(repo.working_dir)

from helper_tools import parser
from helper_tools.base_setup import *

triple_df, entity_df, docs = parser.unified_parser("synthie_text", "test", 10)

Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 40800.62it/s]


Preparing entities for bulk upload to qdrant...


Preparing entities: 100%|██████████| 49/49 [00:01<00:00, 38.61it/s]


No new entities to upload. 49 entities were already in the database.
Preparing predicates for bulk upload to qdrant...


Preparing predicates: 100%|██████████| 32/32 [00:00<00:00, 38.29it/s]

No new predicates to upload. 32 predicates were already in the database.


In [14]:
dataset = []

for i, doc in docs.iterrows():
    doc_triples_df = triple_df[triple_df["docid"] == doc["docid"]]
    expected_triples = ""
    for _, triple in doc_triples_df.iterrows():
        expected_triples += f"({triple['subject']}; {triple['predicate']}; {triple['object']})\n"
    dataset.append({"text":doc["text"], "triples":expected_triples.replace("_"," ")})

In [15]:
from langchain_core.prompts import PromptTemplate


system_prompt = "Please extract all triples (subject; property; object) from the given text. Don't output any additional text, just the triples in this format (subject1; property1; object1)\n(subject2; property2; object2) seperated by line breaks."

prompt_template = PromptTemplate.from_template("""

SYSTEM PROMPT:
{system_prompt}

TEXT:
{text}

""")

response_chain = prompt_template | model

In [16]:
from pydantic import BaseModel

refinement_template = PromptTemplate.from_template("""

Please evaluate the output for the given prompt and give a refinement for the system prompt to match the expected results. Do improve the prompt, so it get incrementally better at extracting triples (not only for the given text). Do not include concrete example, instead keep your adaptions as abstract as possible.

GIVEN PROMPT:
{given_prompt}

OUTPUT:
{output}

EXPECTED OUTPUT:
{expected_output}

""")


class RefinementOutput(BaseModel):
    evaluation: str
    refined_prompt: str

refinement_model = model.with_structured_output(RefinementOutput)

refinement_chain = refinement_template | refinement_model

In [17]:
for i in range(len(dataset)):
    print(f"\n=== Iteration {i} ===")

    # Schritt 1: Generiere Antwort mit aktuellem System-Prompt
    response = response_chain.invoke({"system_prompt": system_prompt, "text": dataset[i]["text"]})
    print(f"\nText {i}:\n{dataset[i]['text']}")
    print(f"\nOutput {i}:\n{response.content}")
    print(f"\nExpected Output {i}:\n{dataset[i]['triples']}")

    # Schritt 2: Prompt-Refinement basierend auf Ausgabe
    refined_input = prompt_template.invoke({"system_prompt": system_prompt, "text": dataset[i]["text"]})
    refinement_response = refinement_chain.invoke({
        "given_prompt": refined_input,
        "output": response.content,
        "expected_output": dataset[i]["triples"]
    })

    # Strukturierte Darstellung des Refinements
    print(f"\n--- Refinement {i} ---")
    print(f"Evaluation:\n{refinement_response.evaluation}")
    print(f"New System Prompt:\n{refinement_response.refined_prompt}")

    # Update System-Prompt für nächste Runde
    system_prompt = refinement_response.refined_prompt

# Letzte finale Ausgabe nach allen Refinements
print(f"\n=== Final Output ===")
response = response_chain.invoke({"system_prompt": system_prompt, "text": dataset[0]["text"]})
print(response.content)
print(f"\nExpected Output:\n{dataset[0]['triples']}")


=== Iteration 0 ===

Text 0:
Groovin' Blue was released on Pacific Jazz Records and performed by Curtis Amy.

Output 0:
<think>
Okay, let's see. The user wants me to extract all triples (subject; property; object) from the given text. The text is: "Groovin' Blue was released on Pacific Jazz Records and performed by Curtis Amy."

First, I need to identify the subjects, properties, and objects here. The main entities are "Groovin' Blue," "Pacific Jazz Records," and "Curtis Amy." 

The first part of the sentence is "Groovin' Blue was released on Pacific Jazz Records." So the subject here is "Groovin' Blue," the property is "released on," and the object is "Pacific Jazz Records." That's one triple.

Then the second part is "performed by Curtis Amy." The subject is still "Groovin' Blue," the property is "performed by," and the object is "Curtis Amy." So that's another triple.

I should check if there are any other possible triples. The sentence structure is straightforward, so I don't thin

In [18]:
for i in range(len(dataset)):
    print(f"\n=== Iteration {i} ===")

    # Schritt 1: Generiere Antwort mit aktuellem System-Prompt
    response = response_chain.invoke({"system_prompt": system_prompt, "text": dataset[i]["text"]})
    print(f"\nText {i}:\n{dataset[i]['text']}")
    print(f"\nOutput {i}:\n{response.content}")
    print(f"\nExpected Output {i}:\n{dataset[i]['triples']}")


=== Iteration 0 ===

Text 0:
Groovin' Blue was released on Pacific Jazz Records and performed by Curtis Amy.

Output 0:
<think>
Okay, let's tackle this query. The user wants me to extract semantic triples from the given text using specific standardized predicates. The example they provided uses 'spouse' for 'married to' and 'sexual orientation' for 'gay man', so I need to make sure I use the most precise noun-based terms.

First, the text is: "Groovin' Blue was released on Pacific Jazz Records and performed by Curtis Amy." 

I need to break this down into subject, property, object. The main entities here are "Groovin' Blue", "Pacific Jazz Records", and "Curtis Amy". 

Starting with "Groovin' Blue was released on Pacific Jazz Records". The action here is 'released on', which in the example is mapped to 'on focus list of' for a similar phrase. But wait, the example used 'on focus list of' for 'is on focus list of', but here it's 'released on'. Maybe the correct predicate is 'record labe